In [2]:
%pip install pydeseq2

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

import pandas as pd

In [4]:
from pathlib import Path
#define project root
project_root = Path.cwd().parent
output_dir = project_root / "quants"
counts = pd.read_csv(output_dir / "GSE60450_merged_raw_counts.txt", sep = "\t")
counts

,GeneID,gene_name,GSM1480291,GSM1480292,GSM1480293,GSM1480294,GSM1480295,GSM1480296,GSM1480297,GSM1480298,GSM1480299,GSM1480300,GSM1480301,GSM1480302
0,100008567,NaN,10,19,0,0,0,4,0,2,2,3,0,0
1,100009600,Zglp1,11,20,15,15,5,1,37,23,23,31,35,20
2,100009609,Vmn2r65,0,0,0,0,0,0,0,0,0,0,0,0
3,100009614,Gm10024,0,0,0,0,0,0,0,0,0,0,0,0
4,100009664,F630206G17Rik,0,0,3,0,0,0,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27174,99889,Arfip1,1157,1189,915,883,627,623,1211,1135,1170,1066,1023,990
27175,99890,Prmt6,464,573,528,423,206,241,376,464,529,400,189,134
27176,99899,Ifi44,43,52,51,39,32,43,22,26,18,14,28,25
27177,99929,Tiparp,2003,2240,1953,1940,1167,935,10230,10363,11175,9418,10280,7883


In [5]:
counts = counts.set_index("GeneID")
counts.dtypes


gene_name     object
GSM1480291     int64
GSM1480292     int64
GSM1480293     int64
GSM1480294     int64
GSM1480295     int64
GSM1480296     int64
GSM1480297     int64
GSM1480298     int64
GSM1480299     int64
GSM1480300     int64
GSM1480301     int64
GSM1480302     int64
dtype: object

In [6]:
counts_with_names = counts.copy()
counts = counts.drop(columns=["gene_name"])
counts = counts[counts.sum(axis=1) > 0]
counts = counts.T
counts

GeneID,100008567,100009600,100009664,100017,100019,100033459,100034251,100034361,100034363,100034684,...,99712,99730,99738,99870,99887,99889,99890,99899,99929,99982
GSM1480291,10,11,0,757,2868,0,7,729,2,10,...,766,582,6,4,9051,1157,464,43,2003,2976
GSM1480292,19,20,0,771,2278,0,8,724,4,18,...,822,640,4,9,9043,1189,573,52,2240,3262
GSM1480293,0,15,3,809,1993,0,24,463,1,3,...,686,458,0,4,739,915,528,51,1953,2532
GSM1480294,0,15,0,813,1572,0,116,407,3,3,...,686,391,0,6,887,883,423,39,1940,2450
GSM1480295,0,5,0,369,568,0,1525,266,1,1,...,764,310,1,1,1024,627,206,32,1167,618
GSM1480296,4,1,0,418,638,1,1665,272,0,1,...,815,312,5,0,1002,623,241,43,935,612
GSM1480297,0,37,1,1162,4162,0,1,585,6,0,...,636,1291,2,7,196,1211,376,22,10230,2745
GSM1480298,2,23,0,1102,2685,0,0,462,6,1,...,599,1092,6,3,196,1135,464,26,10363,2361
GSM1480299,2,23,0,1177,2282,0,1,396,12,0,...,718,1066,6,7,116,1170,529,18,11175,3014
GSM1480300,3,31,0,1064,1947,0,3,352,12,1,...,711,999,4,0,90,1066,400,14,9418,2517


In [7]:
# creat metadata
metadata = pd.DataFrame({
 "sample": [
        "GSM1480291", "GSM1480292",
        "GSM1480293", "GSM1480294",
        "GSM1480295", "GSM1480296",
        "GSM1480297", "GSM1480298",
        "GSM1480299", "GSM1480300",
        "GSM1480301", "GSM1480302",
    ],
    "cell_type": [
        "Luminal", "Luminal",
        "Luminal", "Luminal",
        "Luminal", "Luminal",
        "Basal", "Basal",
        "Basal", "Basal",
        "Basal", "Basal",
    ],
    "stage": [
        "virgin", "virgin",
        "18.5dP", "18.5dP",
        "2dL", "2dL",
        "virgin", "virgin",
        "18.5dP", "18.5dP",
        "2dL", "2dL",
    ],
    "replicate": [
        1, 2,
        1, 2,
        1, 2,
        1, 2,
        1, 2,
        1, 2,
    ]
})
metadata = metadata.set_index("sample")
metadata = metadata.loc[counts.index]
metadata["cell_type"] = pd.Categorical(metadata["cell_type"], categories=["Basal", "Luminal"])
metadata["stage"] = pd.Categorical(metadata["stage"], categories=["virgin", "18.5dP", "2dL"], ordered=True)
metadata.to_csv(project_root / "quants" / "GSE60450_metadata.csv", index=True, index_label="sample")
metadata

,cell_type,stage,replicate
GSM1480291,Luminal,virgin,1
GSM1480292,Luminal,virgin,2
GSM1480293,Luminal,18.5dP,1
GSM1480294,Luminal,18.5dP,2
GSM1480295,Luminal,2dL,1
GSM1480296,Luminal,2dL,2
GSM1480297,Basal,virgin,1
GSM1480298,Basal,virgin,2
GSM1480299,Basal,18.5dP,1
GSM1480300,Basal,18.5dP,2


In [8]:
print(counts.shape)
print(metadata.shape)
print(counts.index.equals(metadata.index))
print(metadata.columns)

(12, 21392)
(12, 3)
True
Index(['cell_type', 'stage', 'replicate'], dtype='object')


In [9]:
# create deseq2 dataset and run deseq2
dds = DeseqDataSet(counts=counts, metadata=metadata, design_factors=["cell_type", "stage"])
dds.deseq2()

Using None as control genes, passed at DeseqDataSet initialization


/usr/lib/python3.12/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/tmp/ipykernel_64498/263658098.py:2: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts, metadata=metadata, design_factors=["cell_type", "stage"])
Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 2.67 seconds.

Fitting dispersion trend curve...
... done in 0.27 seconds.

Fitting MAP dispersions...
... done in 2.90 seconds.

Fitting LFCs...
... done in 2.35 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.



In [10]:
# variance-stabilizing transform for downstream  PCA / clustering 
dds.vst()   # or use rlog equivalent if available
# extract the transformed matrix and convert to DataFrame and save
vsd_df = pd.DataFrame(dds.layers['vst_counts'], index=dds.obs_names, columns=dds.var_names)
vsd_df.to_csv(project_root / "quants" / "GSE60450_vst_matrix.csv", index=True, index_label="sample")
vsd_df.head()

Fitting size factors...
... done in 0.01 seconds.



Fit type used for VST : parametric
Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 2.53 seconds.

/home/r/GitHub_Rong/RNAseq_pipeline/rnaseq_env/lib/python3.12/site-packages/pydeseq2/dds.py:432: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst=True)


GeneID,100008567,100009600,100009664,100017,100019,100033459,100034251,100034361,100034363,100034684,...,99712,99730,99738,99870,99887,99889,99890,99899,99929,99982
GSM1480291,3.059435,3.188579,-1.871937,9.206845,11.127543,-1.871937,2.583341,9.152522,1.049740,3.059435,...,9.223881,8.827972,2.381696,1.866132,12.785335,9.818408,8.501518,5.090850,10.609813,11.180860
GSM1480292,3.857588,3.928916,-1.871937,9.146738,10.708783,-1.871937,2.680429,9.056087,1.791900,3.782546,...,9.239060,8.878364,1.791900,2.837162,12.697465,9.771196,8.719023,5.275623,10.684522,11.226630
GSM1480293,-1.871937,3.914805,1.779420,9.616170,10.916303,-1.871937,4.573433,8.811793,0.550764,1.779420,...,9.378419,8.796147,-1.871937,2.139827,9.485700,9.793686,9.001102,5.643409,10.887062,11.261552
GSM1480294,-1.871937,4.075854,-1.871937,9.789837,10.740681,-1.871937,6.986041,8.792498,1.922608,1.922608,...,9.544955,8.734711,-1.871937,2.820599,9.915442,9.908925,8.848060,5.425636,11.044049,11.380697
GSM1480295,-1.871937,3.219465,-1.871937,9.340332,9.962176,-1.871937,11.386526,8.868608,1.181876,1.181876,...,10.389655,9.089210,1.181876,1.181876,10.812077,10.104676,8.500316,5.825572,11.000612,10.083828


In [11]:
# run deseq2 stats and get results
stat_res = DeseqStats(dds, n_cpus=8, contrast=["cell_type", "Luminal", "Basal"])
stat_res.summary()
res = stat_res.results_df

res

Running Wald tests...


Log2 fold change & Wald test p-value: cell_type Luminal vs Basal
              baseMean  log2FoldChange     lfcSE       stat         pvalue  \
GeneID                                                                       
100008567     2.901077        2.201168  1.526031   1.442414   1.491856e-01   
100009600    18.029326       -1.212957  0.363076  -3.340780   8.354349e-04   
100009664     0.392423        0.393065  3.070669   0.128006   8.981440e-01   
100017      774.075991       -0.276735  0.128169  -2.159147   3.083879e-02   
100019     1781.626379       -0.262751  0.144110  -1.823268   6.826281e-02   
...                ...             ...       ...        ...            ...   
99889       975.337319       -0.031145  0.055441  -0.561770   5.742728e-01   
99890       362.667530        0.557493  0.203273   2.742589   6.095700e-03   
99899        34.599392        1.277820  0.213479   5.985688   2.154768e-09   
99929      5331.353463       -2.290666  0.098985 -23.141621  1.765515e-118   

... done in 2.19 seconds.



,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
GeneID,,,,,,
100008567,2.901077,2.201168,1.526031,1.442414,1.491856e-01,2.001317e-01
100009600,18.029326,-1.212957,0.363076,-3.340780,8.354349e-04,1.723478e-03
100009664,0.392423,0.393065,3.070669,0.128006,8.981440e-01,NaN
100017,774.075991,-0.276735,0.128169,-2.159147,3.083879e-02,4.851816e-02
100019,1781.626379,-0.262751,0.144110,-1.823268,6.826281e-02,9.971963e-02
...,...,...,...,...,...,...
99889,975.337319,-0.031145,0.055441,-0.561770,5.742728e-01,6.363852e-01
99890,362.667530,0.557493,0.203273,2.742589,6.095700e-03,1.094012e-02
99899,34.599392,1.277820,0.213479,5.985688,2.154768e-09,8.791620e-09


In [12]:
# add gene names back to results
gene_names = counts_with_names[["gene_name"]]
gene_names.index = gene_names.index.astype(str)
res_with_names = stat_res.results_df.copy()
res_with_names.index = res_with_names.index.astype(str)

# check index types of counts and stat_res results
print(gene_names.index[:5])
print(res_with_names.index[:5])
print(gene_names.index.dtype)
print(res_with_names.index.dtype)

res_with_names = res_with_names.join(gene_names, how="left")
res_with_names = res_with_names[
    ["gene_name", 'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj']
]
res_with_names

Index(['100008567', '100009600', '100009609', '100009614', '100009664'], dtype='object', name='GeneID')
Index(['100008567', '100009600', '100009664', '100017', '100019'], dtype='object', name='GeneID')
object
object


,gene_name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
GeneID,,,,,,,
100008567,NaN,2.901077,2.201168,1.526031,1.442414,1.491856e-01,2.001317e-01
100009600,Zglp1,18.029326,-1.212957,0.363076,-3.340780,8.354349e-04,1.723478e-03
100009664,F630206G17Rik,0.392423,0.393065,3.070669,0.128006,8.981440e-01,NaN
100017,Ldlrap1,774.075991,-0.276735,0.128169,-2.159147,3.083879e-02,4.851816e-02
100019,Mdn1,1781.626379,-0.262751,0.144110,-1.823268,6.826281e-02,9.971963e-02
...,...,...,...,...,...,...,...
99889,Arfip1,975.337319,-0.031145,0.055441,-0.561770,5.742728e-01,6.363852e-01
99890,Prmt6,362.667530,0.557493,0.203273,2.742589,6.095700e-03,1.094012e-02
99899,Ifi44,34.599392,1.277820,0.213479,5.985688,2.154768e-09,8.791620e-09


In [13]:
# save differntially expressed genes with gene_names to csv
res_with_names.to_csv(project_root / "quants" / "GSE60450_deseq2_results.csv", index=True)

In [14]:
# filter for significant genes with |log2FC| > 1 and padj < 0.05
sig_genes = res_with_names[(res_with_names["padj"] < 0.05) & (res_with_names["log2FoldChange"].abs() > 1)]
sig_genes
# save significant genes to csv
sig_genes.to_csv(project_root / "quants" / "GSE60450_significant_genes.csv", index=True)